<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/denoising_autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow librosa numpy matplotlib

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
import librosa.display

In [ ]:
def load_and_preprocess_audio(file_path):
    # Load audio file
    audio, sr = librosa.load(file_path, sr=22050)  # Set sample rate
    # Convert audio to Mel-spectrogram
    mel_spectrogram = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128, fmax=8000)
    mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)  # Convert to dB scale
    return mel_spectrogram_db, sr

In [ ]:
def plot_mel_spectrogram(mel_spectrogram, sr):
    plt.figure(figsize=(10, 6))
    librosa.display.specshow(mel_spectrogram, x_axis='time', y_axis='mel', sr=sr)
    plt.title('Mel-Spectrogram')
    plt.colorbar(format='%+2.0f dB')
    plt.show()

In [ ]:
# Define the more complex Denoising Autoencoder model
def build_complex_dae(input_shape):
    model = models.Sequential()
    # Encoder
    model.add(layers.InputLayer(input_shape=input_shape))
    model.add(layers.Conv2D(64, (5, 5), activation='LeakyReLU(alpha=0.2)', padding='same'))  # Increased kernel size
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2), padding='same'))
    model.add(layers.Conv2D(128, (5, 5), activation='LeakyReLU(alpha=0.2)', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2), padding='same'))
    model.add(layers.Conv2D(256, (3, 3), activation='LeakyReLU(alpha=0.2)', padding='same'))  # Additional layer
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2), padding='same'))
    # Decoder
    model.add(layers.Conv2DTranspose(256, (3, 3), activation='LeakyReLU(alpha=0.2)', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.UpSampling2D((2, 2)))
    model.add(layers.Conv2DTranspose(128, (5, 5), activation='LeakyReLU(alpha=0.2)', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.UpSampling2D((2, 2)))
    model.add(layers.Conv2DTranspose(64, (5, 5), activation='LeakyReLU(alpha=0.2)', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.UpSampling2D((2, 2)))
    model.add(layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same'))  # Output layer
    return model
# Build the model
input_shape = (128, 130, 1)  # Example Mel-spectrogram dimensions
dae_model = build_complex_dae(input_shape)
# Compile the model with more advanced settings
dae_model.compile(optimizer='adam', loss='mean_squared_error')
# Display the model summary
dae_model.summary()

In [ ]:
# Example arrays (replace with your actual data)
X_train_real = np.random.randn(100, 128, 130, 1)  # Example real data
X_train_ai = np.random.randn(100, 128, 130, 1)  # Example AI-generated data
# Add noise to AI data to simulate the 'AI artifacts'
X_train_noisy = X_train_ai + np.random.normal(0, 0.1, X_train_ai.shape)  # Adding noise

In [ ]:
# Train the Denoising Autoencoder
dae_model.fit(X_train_noisy, X_train_real, epochs=10, batch_size=32)

In [ ]:
# Function to clean up AI-generated music
def clean_up_audio(model, mel_spectrogram):
    mel_spectrogram_input = np.expand_dims(mel_spectrogram, axis=-1)  # Add channel dimension
    mel_spectrogram_input = np.expand_dims(mel_spectrogram_input, axis=0)  # Add batch dimension
    cleaned_mel_spectrogram = model.predict(mel_spectrogram_input)
    return cleaned_mel_spectrogram[0, :, :, 0]
# Example of cleaning up an AI-generated Mel-spectrogram
cleaned_mel_spectrogram = clean_up_audio(dae_model, mel_spectrogram)
# Plot the cleaned Mel-spectrogram
plot_mel_spectrogram(cleaned_mel_spectrogram, sr)

In [ ]:
# Convert cleaned Mel-spectrogram back to audio using Griffin-Lim algorithm
def spectrogram_to_audio(mel_spectrogram_db, sr):
    # Inverse the Mel-spectrogram back to waveform
    mel_spectrogram = librosa.db_to_power(mel_spectrogram_db)
    audio = librosa.feature.inverse.mel_to_audio(mel_spectrogram, sr=sr)
    return audio
# Reconstruct the cleaned audio
cleaned_audio = spectrogram_to_audio(cleaned_mel_spectrogram, sr)
# Play the cleaned audio (in Colab, you can use IPython.display)
import IPython.display as ipd
ipd.Audio(cleaned_audio, rate=sr)

In [ ]:
import soundfile as sf
# Save cleaned audio to file
sf.write('/mnt/data/cleaned_audio.wav', cleaned_audio, sr)